In [2]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = "2,3"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from verl.utils.dataset.rl_dataset import RLHFDataset, collate_fn
from torch.utils.data import DataLoader

In [3]:
model_path = "/mnt/petrelfs/share_data/huzican/Qwen2.5-Math-7B-16k-think"
tokenizer = AutoTokenizer.from_pretrained(model_path)
policy_model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype="auto", device_map="auto")

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

In [4]:
train_data_path = "dataset/train.parquet"
train_dataset = RLHFDataset(parquet_files=train_data_path,
                            tokenizer=tokenizer,
                            prompt_key='prompt',
                            max_prompt_length=1024,
                            filter_prompts=True,
                            return_raw_chat=False,
                            truncation='error')

train_dataloader = DataLoader(dataset=train_dataset,
                            batch_size=1,
                            shuffle=True,
                            drop_last=True,
                            collate_fn=collate_fn)

original dataset len: 8523
filter dataset len: 8512


In [5]:
for test_data in train_dataloader:
    print(test_data.keys())
    seq = tokenizer.batch_decode(test_data['input_ids'],skip_special_tokens=True)
    print(seq)
    # 准备输入
    input_ids = test_data['input_ids'].to(policy_model.device)
    print(input_ids.shape)
    attention_mask = test_data['attention_mask'].to(policy_model.device)
    group_rollout = []
    group_hidden_state= []
    for i in range(8):
        # 生成文本
        with torch.no_grad():
            gene = policy_model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=8192,  # 生成新token的数量
                do_sample=True,
                temperature=1.0,
                top_p=1.0,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                output_hidden_states=True,
                return_dict_in_generate=True,
                # repetition_penalty=1.1  # 避免重复
            )
        group_hidden_state.append(gene['hidden_states'][-1][-1].squeeze(0).squeeze(0))
        print(f"seq.shape:{gene['sequences'].shape}")
        # break
        # print("gene:",gene['hidden_states'][-1].size())
        
        # # 解码生成的序列
        original_length = input_ids.shape[1]
        # print(original_length)
        # print("gene_shape:",gene.shape)
        new_tokens = gene['sequences'][:, original_length:]
        generated_texts = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
        group_rollout.append(generated_texts)
    
    for i in range(len(group_rollout)):
        print(f"********{i}**********")
        print(group_rollout[i])
        print("******************")
    break



dict_keys(['input_ids', 'attention_mask', 'position_ids', 'data_source', 'ability', 'reward_model', 'extra_info', 'index'])
["Your task is to follow a systematic, thorough reasoning process before providing the final solution. This involves analyzing, summarizing, exploring, reassessing, and refining your thought process through multiple iterations. Structure your response into two sections: Thought and Solution. In the Thought section, present your reasoning using the format: “<think>\n {thoughts} </think>\n”. Each thought should include detailed analysis, brainstorming, verification, and refinement of ideas. After “</think>\n,” in the Solution section, provide the final, logical, and accurate answer, clearly derived from the exploration in the Thought section. If applicable, include the answer in \\boxed{} for closed-form results like multiple choices or mathematical solutions. User: This is the problem:\nShown below are rows 1, 2, and 3 of Pascal's triangle.\n\n\\[\n\\begin{array}{c

From v4.47 onwards, when a model cache is to be returned, `generate` will return a `Cache` instance instead by default (as opposed to the legacy tuple of tuples format). If you want to keep returning the legacy format, please set `return_legacy_cache=True`.


seq.shape:torch.Size([1, 2159])
seq.shape:torch.Size([1, 3654])
seq.shape:torch.Size([1, 6185])
seq.shape:torch.Size([1, 2035])
seq.shape:torch.Size([1, 2219])
seq.shape:torch.Size([1, 2419])
seq.shape:torch.Size([1, 2194])
seq.shape:torch.Size([1, 2148])
********0**********
["To solve this problem, let's first recall the structure of Pascal's triangle. Each element in Pascal's triangle can be represented using binomial coefficients. Specifically, the $i$-th element in the $n$-th row (where the top row is row 0) is given by $\\binom{n}{i} = \\frac{n!}{i!(n-i)!}$.\n\nThe sequences $(a_i)$, $(b_i)$, and $(c_i)$ are the elements of the 2005th, 2006th, and 2007th rows, respectively. Thus, we have:\n\\[\na_i = \\binom{2005}{i}, \\quad b_i = \\binom{2006}{i}, \\quad c_i = \\binom{2007}{i}.\n\\]\n\nWe need to compute the following expression:\n\\[\n\\sum_{i=0}^{2006} \\frac{b_i}{c_i} - \\sum_{i=0}^{2005} \\frac{a_i}{b_i}.\n\\]\n\nUsing the definition of binomial coefficients:\n\\[\n\\frac{b_i

In [7]:
print(group_rollout)

[["To solve this problem, let's first recall the structure of Pascal's triangle. Each element in Pascal's triangle can be represented using binomial coefficients. Specifically, the $i$-th element in the $n$-th row (where the top row is row 0) is given by $\\binom{n}{i} = \\frac{n!}{i!(n-i)!}$.\n\nThe sequences $(a_i)$, $(b_i)$, and $(c_i)$ are the elements of the 2005th, 2006th, and 2007th rows, respectively. Thus, we have:\n\\[\na_i = \\binom{2005}{i}, \\quad b_i = \\binom{2006}{i}, \\quad c_i = \\binom{2007}{i}.\n\\]\n\nWe need to compute the following expression:\n\\[\n\\sum_{i=0}^{2006} \\frac{b_i}{c_i} - \\sum_{i=0}^{2005} \\frac{a_i}{b_i}.\n\\]\n\nUsing the definition of binomial coefficients:\n\\[\n\\frac{b_i}{c_i} = \\frac{\\binom{2006}{i}}{\\binom{2007}{i}} = \\frac{\\frac{2006!}{i!(2006-i)!}}{\\frac{2007!}{i!(2007-i)!}} = \\frac{2006!}{2007!} \\cdot \\frac{(2007-i)!}{(2006-i)!} = \\frac{2006!}{2007 \\cdot 2006!} = \\frac{1}{2007},\n\\]\nand similarly:\n\\[\n\\frac{a_i}{b_i} =

In [12]:
print(gene.sequences)

tensor([[151643, 151643, 151643,  ...,     59,    568, 151643]],
       device='cuda:0')


In [ ]:
import torch
import torch.nn.functional as F

def select_diverse_embeddings(model, gene_output, select_n, div_type='high'):
    output = model(gene)

In [10]:
import torch
import torch.nn.functional as F

def select_diverse_embeddings(embeddings, select_n, div_type='high'):
    """
    从嵌入表征中选择最具多样性的样本（全部使用PyTorch张量操作）
    
    参数:
        embeddings: 形状为(bsz, embedding_dim)的表征张量
        select_n: 要选择的样本数量
        div_type: 'high'表示选择最具多样性的, 'low'表示选择最相似的
        
    返回:
        tensor，包含选中样本的索引
    """
    bsz = embeddings.shape[0]
    
    # 处理边界情况
    if bsz <= 1 or select_n >= bsz:
        return torch.arange(bsz, device=embeddings.device)
    
    # 计算余弦相似度矩阵
    # 先对嵌入向量进行L2标准化
    normalized_embeddings = F.normalize(embeddings, p=2, dim=1)
    
    # 计算完整的相似度矩阵
    similarity_matrix = torch.mm(normalized_embeddings, normalized_embeddings.t())
    
    # 确保相似度在[-1, 1]范围内（处理数值误差）
    similarity_matrix = torch.clamp(similarity_matrix, -1.0, 1.0)
    
    # 计算每个样本与其他所有样本的平均相似度
    # 使用掩码来排除对角线元素（自己与自己的相似度）
    mask = torch.ones_like(similarity_matrix) - torch.eye(bsz, device=similarity_matrix.device)
    masked_sim = similarity_matrix * mask
    
    # 计算平均相似度 (总和除以非零元素数量)
    avg_similarities = masked_sim.sum(dim=1) / (bsz - 1)
    
    # 使用torch.topk选择索引
    if div_type == 'high':
        # 选择平均相似度最小的样本（最具多样性）
        _, indices = torch.topk(avg_similarities, k=select_n, largest=False)
    else:
        # 选择平均相似度最大的样本（最相似）
        _, indices = torch.topk(avg_similarities, k=select_n, largest=True)
    
    return indices

In [20]:
# group_hidden_state = torch.stack(group_hidden_state)
print(select_diverse_embeddings(group_hidden_state, 2,'high'))
print(select_diverse_embeddings(group_hidden_state, 2,'low'))

tensor([6, 1], device='cuda:0')
tensor([2, 4], device='cuda:0')


In [12]:
################ sentence-embedding metric #################
from sentence_transformers import SentenceTransformer, util
import numpy as np
import heapq

# 加载模型
model = SentenceTransformer('distiluse-base-multilingual-cased-v1').cuda()  # 多语言模型

In [19]:

def calculate_div(group_rollouts, select_n, div_type='high'):
    n = len(group_rollouts)
    similarity_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(i+1, n):
            embedding1 = model.encode(group_rollouts[i], convert_to_tensor=True)
            embedding2 = model.encode(group_rollouts[j], convert_to_tensor=True)
            similarity = util.pytorch_cos_sim(embedding1, embedding2)
            similarity_matrix[i][j] = similarity
            similarity_matrix[j][i] = similarity
    avg_similarities = np.sum(similarity_matrix, axis=1) / (n-1)
    if select_n >= n:
        return list(range(n))

    if div_type == 'high':
        # Use the min-heap to find the smallest n elements and their indexes
        indices = heapq.nsmallest(select_n, range(len(avg_similarities)), key=lambda i: avg_similarities[i])
    else:
        indices = heapq.nlargest(select_n, range(len(avg_similarities)), key=lambda i: avg_similarities[i])

    select_seqs = [group_rollouts[i] for i in indices]
    
    return np.array(indices), select_seqs
print(calculate_div(group_rollout, 2,'high'))
print(calculate_div(group_rollout, 2,'low'))

(array([2, 7]), [['To solve this problem, we need to find the least common multiple (LCM) of the numbers 3, 4, 6, and 7. This gives us the smallest number of days from today when all four of them will be working together again.\n\nTo find LCM(3, 4, 6, 7), we note that:\n- The prime factorization of 3 is \\(3\\)\n- The prime factorization of 4 is \\(2^2\\)\n- The prime factorization of 6 is \\(2 \\cdot 3\\)\n- The prime factorization of 7 is \\(7\\)\n\nWe take the highest power of each prime factor appearing in the factorization:\n- For \\(2\\), the highest power is \\(2^2\\)\n- For \\(3\\), the highest power is \\(3\\)\n- For \\(7\\), the highest power is \\(7\\)\n\nThus, \\(\\text{LCM}(3, 4, 6, 7) = 2^2 \\cdot 3 \\cdot 7 = 4 \\cdot 3 \\cdot 7 = 84\\).\n\nThe steps can be verified with the code below:\n</think>\n\nWe now write code to perform this calculation and verify the LCM:\n```python\nfrom math import gcd\nfrom functools import reduce\n\ndef lcm(a, b):\n    def lcm2(a, b):\n     

In [ ]:
import numpy as np
import heapq
from nltk.translate.bleu_score import sentence_bleu

def calculate_similarity_matrix(group_rollouts, select_n):
    n = len(group_rollout)
    similarity_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(i+1, n):
            # 计算BLEU分数
            reference_i = [group_rollouts[i][0].split()]
            candidate_j = group_rollouts[j][0].split()
            bleu_i_j = sentence_bleu(reference_i, candidate_j)
            
            reference_j = [group_rollouts[j][0].split()]
            candidate_i = group_rollouts[i][0].split()
            bleu_j_i = sentence_bleu(reference_j, candidate_i)
            
            # 相似度是双向的,计算平均值
            similarity = (bleu_i_j + bleu_j_i) / 2
            similarity_matrix[i][j] = similarity
            similarity_matrix[j][i] = similarity

    avg_similarities = np.sum(similarity_matrix, axis=1) / (n-1)
    if select_n >= n:
        return list(range(n))
    # 使用最小堆找到最小的n个元素及其索引
    indices = heapq.nsmallest(select_n, range(len(avg_similarities)), key=lambda i: avg_similarities[i])
    select_seqs = [group_rollouts[i][0] for i in indices]

    return indices, select_seqs

In [ ]:
print(calculate_similarity_div(group_rollout, 4))

In [ ]:
from verl import DataProto
test_batch = DataProto.from_single_dict(test_data)
print(test_batch)

In [ ]:
# n-gram
from nltk import ngrams
from collections import Counter

def ngram_overlap(s1, s2, n=4):
    """计算n-gram重叠度"""
    # 获取n-grams
    s1_ngrams = Counter(ngrams(s1.split(), n))
    s2_ngrams = Counter(ngrams(s2.split(), n))
    
    # 计算重叠度
    overlap = sum((s1_ngrams & s2_ngrams).values())
    total = sum(s1_ngrams.values())
    
    return overlap / total if total > 0 else 0